# Notebook 08 — How the Model Reads: Tokens and the Four Dials (Lesson 12)

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com then File > Upload notebook and choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths.

The model never sees your words — it sees **tokens** (words or word-pieces). This notebook
lets you interrogate the real tokenizer, then reads off the four dials of the model's spec
sheet. One activity per cell.

In [ ]:
# Run once: load the model (its tokenizer comes with it).
%pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer  # the library that gives us the model and its tokenizer
model = SentenceTransformer("all-MiniLM-L6-v2")  # downloads the model once, then loads it from cache
print("Ready.")  # prints when the model has finished loading

## Tokenize ordinary words
This cell runs five words through the tokenizer and prints the pieces each one breaks into. Common words stay whole: `the` comes back as `['the']`, one piece. Rarer words snap together from smaller known pieces. `unhappiness` comes back as four pieces `['un', '##ha', '##pp', '##iness']`, and `antidisestablishmentarianism` breaks into eight. The `##` prefix means the piece glues onto the one before it with no space. The count in brackets is the number of tokens, which is what the model actually counts.

In [2]:
# Activity: watch words split into tokens.
for w in ["the", "running", "transformer", "unhappiness", "antidisestablishmentarianism"]:
    pieces = model.tokenizer.tokenize(w)  # ask the model's tokenizer to split the word into token pieces
    print(f"{w:<30} {pieces}   ({len(pieces)} tokens)")  # ## on a piece means it glues onto the one before; len counts tokens

the                            ['the']   (1 tokens)
running                        ['running']   (1 tokens)
transformer                    ['transform', '##er']   (2 tokens)
unhappiness                    ['un', '##ha', '##pp', '##iness']   (4 tokens)
antidisestablishmentarianism   ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism']   (8 tokens)


## Tokenize weird real-world text
The same tokenizer has to handle anything you throw at it. This cell feeds it a virus name, a shorthand, a contraction, an accented word, a made-up product name, and a URL. Watch what happens: `COVID-19` splits into `['co', '##vid', '-', '19']`, the accent in `naïve` is dropped down to `naive`, and the URL is chopped into many small pieces, with every slash, dot, and symbol becoming its own token.

In [3]:
# Activity: weird inputs through the same tokenizer.
for s in ["COVID-19", "k8s", "don't", "naïve", "ZorpTab90",
          "https://example.com/path?id=42"]:
    print(f"{s:<34} {model.tokenizer.tokenize(s)}")  # same tokenize call; punctuation and symbols each become their own token

COVID-19                           ['co', '##vid', '-', '19']
k8s                                ['k', '##8', '##s']
don't                              ['don', "'", 't']
naïve                              ['naive']
ZorpTab90                          ['z', '##or', '##pta', '##b', '##90']
https://example.com/path?id=42     ['https', ':', '/', '/', 'example', '.', 'com', '/', 'path', '?', 'id', '=', '42']


## Count tokens, not words
Limits and storage budgets are measured in tokens, so it helps to count them. This cell wraps `tokenize(...)` in `len(...)` to count the pieces in a whole sentence. The nine-word fox sentence comes out as 10 tokens because the full stop counts as its own token. The second sentence reaches 14 tokens because `k8s` and `COVID-19` each split into several pieces.

In [4]:
# Activity: token counts for whole sentences.
for s in ["The quick brown fox jumps over the lazy dog.",
          "I tested k8s on the COVID-19 dataset."]:
    n = len(model.tokenizer.tokenize(s))  # len of the token list counts tokens, not words (the full stop counts too)
    print(f"{n:>3} tokens   {s}")  # print the token count next to the sentence

 10 tokens   The quick brown fox jumps over the lazy dog.
 14 tokens   I tested k8s on the COVID-19 dataset.


## The four dials
Every embedding model has a short spec sheet: which tokenizer it uses, how many tokens it knows (vocabulary size), the longest input it accepts (max sequence length), and how many numbers each vector holds (dimension). This cell reads three of those straight off our model. Expect a vocabulary of 30522 tokens, a maximum of 256 tokens per input, and a dimension of 384.

In [5]:
# Activity: read the model's own spec sheet.
print("vocabulary size   :", model.tokenizer.vocab_size)  # how many distinct tokens the tokenizer knows
print("max sequence length:", model.max_seq_length, "tokens")  # longest input the model accepts, in tokens
print("dimension          :", model.get_sentence_embedding_dimension())  # how many numbers are in each output vector

vocabulary size   : 30522
max sequence length: 256 tokens
dimension          : 384


/var/folders/c4/07hkq_hs7h30294jld4f9xx40000gn/T/ipykernel_6976/2097885776.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("dimension          :", model.get_sentence_embedding_dimension())  # how many numbers are in each output vector


In [6]:
# Activity: the storage bill — dial 4 in megabytes.
docs = 100_000  # pretend we are storing one vector for each of 100,000 documents
dim = model.get_sentence_embedding_dimension()  # 384 numbers per vector
mb = docs * dim * 4 / 1024 / 1024  # 4 bytes per number; divide by 1024 twice to go bytes -> KB -> MB
print(f"{docs:,} documents x {dim} numbers x 4 bytes = {mb:.0f} MB of vectors")  # the total storage bill

100,000 documents x 384 numbers x 4 bytes = 146 MB of vectors


/var/folders/c4/07hkq_hs7h30294jld4f9xx40000gn/T/ipykernel_6976/3679280175.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = model.get_sentence_embedding_dimension()  # 384 numbers per vector


You can now read any embedding model's spec sheet. Notebook 09 shows the five ways
these tools can fool you — and the defence for each.

## Practice — Your Turn
Three short exercises to make the tokenizer feel concrete. Read each one, make a guess on paper, then run the answer cell below it to check yourself.

### Exercise 1 — How many pieces?
Pick three words: a common everyday word, a long rare word, and your own name. Before you run anything, write down how many tokens you think each one becomes. A common word is usually 1 token. A long rare word often splits into several pieces, each marked with `##`. Names can surprise you.

Replace the words in the answer cell with your own, then compare the word count (always 1 here, since each is a single word) against the token count.

Try it yourself, then run the answer cell below.

In [7]:
# Answer
words = ["river", "antidisestablishmentarianism", "Ritesh"]  # a common word, a long rare word, your name
for w in words:                                              # go through each word one at a time
    pieces = model.tokenizer.tokenize(w)                     # split the word into token pieces (## glues onto the piece before)
    word_count = len(w.split())                              # word count: a single word is 1
    token_count = len(pieces)                                # token count: how many pieces the model actually sees
    print(f"{w:<32} {pieces}")                               # show the word and the pieces it broke into
    print(f"{'':<32} {word_count} word -> {token_count} tokens")  # show the gap between words and tokens
    print()                                                  # blank line between words for readability

river                            ['river']
                                 1 word -> 1 tokens

antidisestablishmentarianism     ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism']
                                 1 word -> 8 tokens

Ritesh                           ['rites', '##h']
                                 1 word -> 2 tokens



### Exercise 2 — Find an expensive short phrase
A phrase can be only three words long and still cost a pile of tokens. Digits, hyphens, slashes, and other punctuation each tend to become their own token. Think of a product code, a version string, or a short URL.

Guess how many tokens the phrase below costs, then run the answer cell. Notice how far the token count climbs above the word count.

Try it yourself, then run the answer cell below.

In [8]:
# Answer
phrase = "ZorpTab90 v2.3-beta https://x.io/p?id=7"   # three "words" stuffed with digits, hyphens, dots, slashes
words = phrase.split()                                # split on spaces to get the word count
pieces = model.tokenizer.tokenize(phrase)            # split the whole phrase into token pieces
print("phrase     :", phrase)                         # the phrase we are measuring
print("word count :", len(words))                     # how many space-separated words it has
print("token count:", len(pieces))                    # how many tokens the model charges for it
print("pieces     :", pieces)                          # the actual pieces, so you can see each symbol become its own token

phrase     : ZorpTab90 v2.3-beta https://x.io/p?id=7
word count : 3
token count: 24
pieces     : ['z', '##or', '##pta', '##b', '##90', 'v', '##2', '.', '3', '-', 'beta', 'https', ':', '/', '/', 'x', '.', 'io', '/', 'p', '?', 'id', '=', '7']


### Exercise 3 — Work out the storage bill
Each vector from this model holds 384 numbers, and each number takes 4 bytes. So one vector costs 384 x 4 bytes. Store a vector for every document and the bill grows with the number of documents.

Predict roughly how many megabytes 50,000 documents will need, then run the answer cell to see the arithmetic step by step. This is pure arithmetic, so it runs even if the model is loaded.

Try it yourself, then run the answer cell below.

In [9]:
# Answer
dim = model.get_sentence_embedding_dimension()   # 384 numbers per vector for this model
bytes_per_number = 4                              # each number is stored as 4 bytes
docs = 50_000                                     # how many documents we want to store a vector for

one_vector = dim * bytes_per_number               # bytes for a single vector: 384 x 4
total_bytes = one_vector * docs                   # bytes for every document's vector
total_mb = total_bytes / 1024 / 1024              # divide by 1024 twice to go bytes -> KB -> MB

print(f"one vector  : {dim} numbers x {bytes_per_number} bytes = {one_vector:,} bytes")  # cost of a single vector
print(f"total bytes : {one_vector:,} x {docs:,} docs = {total_bytes:,} bytes")           # cost for all documents
print(f"total       : {total_mb:.1f} MB")                                                # the same number in megabytes

one vector  : 384 numbers x 4 bytes = 1,536 bytes
total bytes : 1,536 x 50,000 docs = 76,800,000 bytes
total       : 73.2 MB


/var/folders/c4/07hkq_hs7h30294jld4f9xx40000gn/T/ipykernel_6976/2280257145.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = model.get_sentence_embedding_dimension()   # 384 numbers per vector for this model
